# The Pauli gates X, Y and Z
**Quantum Computing · Week 2 (Gates and the Bloch sphere) · Day 2**

Lesson page: https://sara-aminpour.com/tutorials/qc/w2d2.html

In [ ]:
# On Google Colab, this installs qiskit (about a minute). On your own computer, skip it and
# install the packages once instead: pip install qiskit==2.5.2 scikit-learn torch matplotlib
import sys
if "google.colab" in sys.modules:
    %pip install --quiet qiskit==2.5.2

The three Pauli gates X, Y and Z: as matrices, as actions on states, and as half-turns of the Bloch sphere.

In [ ]:
import numpy as np
from qiskit.quantum_info import Operator, Statevector

X, Y, Z = (Operator.from_label(p).data for p in "XYZ")
I = np.eye(2)
for name, M in [("X", X), ("Y", Y), ("Z", Z)]:
    print(f"{name} =\n{np.round(M, 3)}\n")
print("X·X = I?", np.allclose(X @ X, I), "  Y·Y = I?", np.allclose(Y @ Y, I), "  Z·Z = I?", np.allclose(Z @ Z, I))
print("X·Z = -Z·X?", np.allclose(X @ Z, -Z @ X), "   Y = i·X·Z?", np.allclose(Y, 1j * X @ Z))

## What they do to the famous states
`ket` prints a state in Dirac notation. Read each line, and predict the next one before you look.

In [ ]:
def fmt(z):
    re, im = round(z.real, 3), round(z.imag, 3)
    if im == 0:
        return f"{re:g}"
    if re == 0:
        return f"{im:g}i"
    return f"({re:g}{im:+g}i)"

def ket(sv):
    out = ""
    for amp, label in zip(sv.data, ["|0>", "|1>"]):
        if abs(amp) < 1e-9:
            continue
        c = fmt(amp)
        if not out:
            out = c + label
        elif c.startswith("-"):
            out += " - " + c[1:] + label
        else:
            out += " + " + c + label
    return out

s = 1 / np.sqrt(2)
famous = {"|0>": [1, 0], "|1>": [0, 1], "|+>": [s, s], "|->": [s, -s]}
for gname, G in [("X", X), ("Z", Z), ("Y", Y)]:
    for sname, amps in famous.items():
        out = Statevector(amps).evolve(Operator(G))
        print(f"{gname} {sname:>3} = {ket(out)}")
    print()

## Z is invisible to a 0/1 measurement
Z changes $|+\rangle$ into $|-\rangle$: different states, same 0/1 odds.

In [ ]:
plus = Statevector(famous["|+>"])
after = plus.evolve(Operator(Z))
print("P(0), P(1) before:", plus.probabilities().round(3), "  after:", after.probabilities().round(3))
print("overlap |<before|after>|^2 =", round(abs(plus.inner(after)) ** 2, 3), " -> completely different states")

## On the Bloch sphere: half-turns
A half-turn about the x axis keeps $x$ and flips the signs of $y$ and $z$; and so on.

In [ ]:
def bloch(sv):
    a, b = sv.data
    c = np.conj(a) * b
    return np.round([2 * c.real, 2 * c.imag, abs(a) ** 2 - abs(b) ** 2], 3)

psi = Statevector([np.cos(0.4), np.exp(0.9j) * np.sin(0.4)])
print("start        ", bloch(psi))
for gname, G in [("X", X), ("Y", Y), ("Z", Z)]:
    print(f"after {gname}      ", bloch(psi.evolve(Operator(G))))

## Your turn: who stays put?
For each Pauli gate, find which of the six famous states it leaves unchanged **up to a global phase**.
`Statevector.equiv` compares states while ignoring a global phase.

In [ ]:
six = {**famous, "|+i>": [s, 1j * s], "|-i>": [s, -1j * s]}
for gname, G in [("X", X), ("Y", Y), ("Z", Z)]:
    stay = []
    for sname, amps in six.items():
        sv = Statevector(amps)
        # TODO: if applying G leaves sv unchanged up to a global phase, add sname to `stay`
    print(f"{gname} leaves alone: {stay}")

---
## Solution
Try the exercise yourself first: it's fine to get stuck for a minute. Then compare with this one (there are many good answers).

In [ ]:
for gname, G in [("X", X), ("Y", Y), ("Z", Z)]:
    stay = [sname for sname, amps in six.items() if Statevector(amps).evolve(Operator(G)).equiv(Statevector(amps))]
    print(f"{gname} leaves alone: {stay}")
print("Each gate leaves exactly the two states on its own axis: these are its eigenstates.")